# Student Zero-Shot and LoRA SFT Experiments

Use this notebook to run the next project stage: validate teacher data, evaluate the base student zero-shot, build SFT files, train one or more LoRA configs, and compare output metrics.

Recommended flow: run zero-shot first, inspect metrics and outputs, then enable one small SFT config before running a larger config.

## 1. Optional Colab Pull

Run this cell in Colab when you want a fresh copy from GitHub. Skip it when running locally from the repo.

In [ ]:
# Optional: uncomment in Colab.
# from pathlib import Path
# REPO_URL = "https://github.com/kdnehihi/strategy-distill-rl.git"
# REPO_DIR = Path("/content/strategy-distill-rl")
# if REPO_DIR.exists():
#     !git -C /content/strategy-distill-rl pull origin main
# else:
#     !git clone {REPO_URL} /content/strategy-distill-rl
# %cd /content/strategy-distill-rl

## 2. Install Dependencies

Run this once per fresh environment. Local machines that already have the environment can skip it.

In [ ]:
# Optional: uncomment on a fresh Colab/runtime.
# !pip install -r requirements.txt

## 3. Experiment Config

Change the flags and config list here. Keep the first run small; then increase samples or epochs once the output format looks stable.

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()

MODEL_NAME = "Qwen/Qwen2.5-Math-1.5B-Instruct"
TEACHER_PATH = "data/gsm8k_teacher_preview_5000.jsonl"
EVAL_INPUT_PATH = "data/gsm8k_clean_test.jsonl"
SFT_TRAIN_PATH = "data/sft_strategy_train.jsonl"
SFT_VAL_PATH = "data/sft_strategy_val.jsonl"

RUN_VALIDATE_TEACHER = True
RUN_ZERO_SHOT = True
RUN_BUILD_SFT = True
RUN_TRAINING = False
RUN_ADAPTER_EVAL = True

ZERO_SHOT_NUM_SAMPLES = 100
FINAL_EVAL_NUM_SAMPLES = 100
EVAL_BATCH_SIZE = 8
EVAL_MAX_NEW_TOKENS = 256

TRAIN_CONFIGS = [
    {
        "name": "smoke_r8_a16_300",
        "max_train_samples": 300,
        "max_val_samples": 100,
        "epochs": 1.0,
        "lr": 2e-4,
        "lora_r": 8,
        "lora_alpha": 16,
        "lora_dropout": 0.05,
        "grad_accum": 8,
    },
    {
        "name": "balanced_r16_a32_4000",
        "max_train_samples": 4000,
        "max_val_samples": 298,
        "epochs": 1.0,
        "lr": 2e-4,
        "lora_r": 16,
        "lora_alpha": 32,
        "lora_dropout": 0.05,
        "grad_accum": 8,
    },
    {
        "name": "stronger_r32_a64_4000",
        "max_train_samples": 4000,
        "max_val_samples": 298,
        "epochs": 2.0,
        "lr": 1e-4,
        "lora_r": 32,
        "lora_alpha": 64,
        "lora_dropout": 0.05,
        "grad_accum": 8,
    },
]

RUNS_DIR = Path("runs/student_sft")
CHECKPOINTS_DIR = Path("checkpoints/student_sft")
RUNS_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

## 4. Helpers

These helpers run repo scripts and load metric JSON files into a comparison table.

In [ ]:
import json
import subprocess
from pathlib import Path

import pandas as pd


def run_command(args):
    print("$", " ".join(str(arg) for arg in args))
    subprocess.run([str(arg) for arg in args], check=True)


def load_json(path):
    with Path(path).open("r", encoding="utf-8") as f:
        return json.load(f)


def read_jsonl(path, limit=None):
    rows = []
    with Path(path).open("r", encoding="utf-8") as f:
        for line in f:
            if limit is not None and len(rows) >= limit:
                break
            rows.append(json.loads(line))
    return rows


def metric_row(name, metrics_path, train_metrics_path=None):
    metrics = load_json(metrics_path)
    row = {
        "run": name,
        "total": metrics.get("total"),
        "accuracy": metrics.get("accuracy"),
        "format_valid_rate": metrics.get("format_valid_rate"),
        "usable_rate": metrics.get("usable_rate"),
        "correct": metrics.get("correct"),
        "format_valid": metrics.get("format_valid"),
        "usable": metrics.get("usable"),
        "metrics_path": str(metrics_path),
    }
    if train_metrics_path and Path(train_metrics_path).exists():
        train_metrics = load_json(train_metrics_path)
        row["eval_loss"] = train_metrics.get("eval", {}).get("eval_loss")
        row["train_loss"] = train_metrics.get("train", {}).get("train_loss")
    return row

## 5. Validate Teacher Data and Build SFT Files

The teacher file should already contain only usable records. Validation is still the gate before student training.

In [ ]:
if RUN_VALIDATE_TEACHER:
    if not Path(TEACHER_PATH).exists():
        raise FileNotFoundError(
            f"Missing {TEACHER_PATH}. Upload or copy your validated teacher JSONL first."
        )
    run_command([
        "python", "-B", "scripts/validate_teacher_dataset.py",
        "--path", TEACHER_PATH,
    ])

if RUN_BUILD_SFT:
    run_command([
        "python", "-B", "scripts/build_sft_dataset.py",
        "--teacher-path", TEACHER_PATH,
        "--train-output", SFT_TRAIN_PATH,
        "--val-output", SFT_VAL_PATH,
        "--train-size", "4000",
    ])

## 6. Zero-Shot Baseline

Metrics used here are the most important for this project stage: answer exact match, strict format-valid rate, and usable rate. Usable means both correct answer and valid final-block format.

In [ ]:
baseline_output = RUNS_DIR / "zero_shot_outputs.jsonl"
baseline_metrics = RUNS_DIR / "zero_shot_metrics.json"

if RUN_ZERO_SHOT:
    run_command([
        "python", "-B", "scripts/evaluate_student.py",
        "--model-name", MODEL_NAME,
        "--input-path", EVAL_INPUT_PATH,
        "--output-path", baseline_output,
        "--metrics-path", baseline_metrics,
        "--num-samples", str(ZERO_SHOT_NUM_SAMPLES),
        "--batch-size", str(EVAL_BATCH_SIZE),
        "--max-new-tokens", str(EVAL_MAX_NEW_TOKENS),
    ])

if baseline_metrics.exists():
    display(pd.DataFrame([metric_row("zero_shot", baseline_metrics)]))

## 7. Inspect Zero-Shot Outputs

Look at a few failed or invalid outputs before training. If zero-shot already follows the format well, keep SFT lighter.

In [ ]:
if baseline_output.exists():
    rows = read_jsonl(baseline_output)
    failed = [row for row in rows if not row.get("is_usable")]
    print(f"total={len(rows)} failed_or_unusable={len(failed)}")
    for row in failed[:3]:
        print("=" * 80)
        print("id:", row["id"])
        print("question:", row["question"])
        print("ground_truth:", row["ground_truth"])
        print("model_answer:", row["model_answer"])
        print("is_correct:", row["is_correct"], "is_format_valid:", row["is_format_valid"])
        print("format_checks:", row["format_checks"])
        print("raw_model_output:")
        print((row.get("raw_model_output") or "")[:1600])

## 8. Train LoRA SFT Configs

Set `RUN_TRAINING = True` in the config cell when ready. Start with `smoke_r8_a16_300`; only run the larger configs after the smoke run produces sane outputs.

In [ ]:
trained_configs = []

if RUN_TRAINING:
    for cfg in TRAIN_CONFIGS:
        output_dir = CHECKPOINTS_DIR / cfg["name"]
        run_command([
            "python", "-B", "scripts/train_sft_lora.py",
            "--model-name", MODEL_NAME,
            "--train-path", SFT_TRAIN_PATH,
            "--val-path", SFT_VAL_PATH,
            "--output-dir", output_dir,
            "--max-train-samples", str(cfg["max_train_samples"]),
            "--max-val-samples", str(cfg["max_val_samples"]),
            "--num-train-epochs", str(cfg["epochs"]),
            "--learning-rate", str(cfg["lr"]),
            "--lora-r", str(cfg["lora_r"]),
            "--lora-alpha", str(cfg["lora_alpha"]),
            "--lora-dropout", str(cfg["lora_dropout"]),
            "--gradient-accumulation-steps", str(cfg["grad_accum"]),
        ])
        trained_configs.append({**cfg, "output_dir": output_dir})
else:
    for cfg in TRAIN_CONFIGS:
        output_dir = CHECKPOINTS_DIR / cfg["name"]
        if output_dir.exists():
            trained_configs.append({**cfg, "output_dir": output_dir})

print("Adapters available for eval:")
for cfg in trained_configs:
    print(cfg["name"], "->", cfg["output_dir"])

## 9. Evaluate Trained Adapters

This uses the exact same evaluator as zero-shot, so the comparison is apples-to-apples.

In [ ]:
adapter_eval_rows = []

if RUN_ADAPTER_EVAL:
    for cfg in trained_configs:
        name = cfg["name"]
        adapter_path = cfg["output_dir"]
        output_path = RUNS_DIR / f"{name}_outputs.jsonl"
        metrics_path = RUNS_DIR / f"{name}_metrics.json"
        train_metrics_path = adapter_path / "train_metrics.json"

        run_command([
            "python", "-B", "scripts/evaluate_student.py",
            "--model-name", MODEL_NAME,
            "--adapter-path", adapter_path,
            "--input-path", EVAL_INPUT_PATH,
            "--output-path", output_path,
            "--metrics-path", metrics_path,
            "--num-samples", str(FINAL_EVAL_NUM_SAMPLES),
            "--batch-size", str(EVAL_BATCH_SIZE),
            "--max-new-tokens", str(EVAL_MAX_NEW_TOKENS),
        ])
        adapter_eval_rows.append(metric_row(name, metrics_path, train_metrics_path))

adapter_eval_rows

## 10. Compare Metrics

Prioritize `usable_rate` first, then `accuracy`, then `format_valid_rate`. A good SFT config should improve correctness without breaking the strict output contract.

In [ ]:
comparison_rows = []
if baseline_metrics.exists():
    comparison_rows.append(metric_row("zero_shot", baseline_metrics))

for cfg in TRAIN_CONFIGS:
    name = cfg["name"]
    metrics_path = RUNS_DIR / f"{name}_metrics.json"
    train_metrics_path = CHECKPOINTS_DIR / name / "train_metrics.json"
    if metrics_path.exists():
        comparison_rows.append(metric_row(name, metrics_path, train_metrics_path))

comparison = pd.DataFrame(comparison_rows)
if not comparison.empty:
    display(
        comparison.sort_values(
            by=["usable_rate", "accuracy", "format_valid_rate"],
            ascending=False,
        )
    )
    comparison.to_csv(RUNS_DIR / "comparison.csv", index=False)
    print(f"Saved comparison to {RUNS_DIR / 'comparison.csv'}")
else:
    print("No metrics found yet.")

## 11. Paired Compare

Compare zero-shot and each LoRA adapter on the exact same example ids. This is more useful than aggregate accuracy alone because it shows which run fixed baseline mistakes and which run regressed previously correct answers.

In [ ]:
def paired_compare_outputs(base_path, adapter_path, run_name):
    base_rows = read_jsonl(base_path)
    adapter_rows = read_jsonl(adapter_path)

    base_by_id = {row["id"]: row for row in base_rows}
    adapter_by_id = {row["id"]: row for row in adapter_rows}
    common_ids = sorted(set(base_by_id) & set(adapter_by_id))

    paired_rows = []
    for example_id in common_ids:
        base = base_by_id[example_id]
        adapter = adapter_by_id[example_id]
        base_correct = int(base.get("is_correct", 0))
        adapter_correct = int(adapter.get("is_correct", 0))
        base_format = int(base.get("is_format_valid", 0))
        adapter_format = int(adapter.get("is_format_valid", 0))
        base_usable = int(base.get("is_usable", 0))
        adapter_usable = int(adapter.get("is_usable", 0))

        if base_correct and adapter_correct:
            outcome = "both_correct"
        elif (not base_correct) and adapter_correct:
            outcome = "adapter_only"
        elif base_correct and (not adapter_correct):
            outcome = "baseline_only"
        else:
            outcome = "both_wrong"

        paired_rows.append({
            "run": run_name,
            "id": example_id,
            "outcome": outcome,
            "question": base.get("question"),
            "ground_truth": base.get("ground_truth"),
            "baseline_answer": base.get("model_answer"),
            "adapter_answer": adapter.get("model_answer"),
            "baseline_correct": base_correct,
            "adapter_correct": adapter_correct,
            "baseline_format_valid": base_format,
            "adapter_format_valid": adapter_format,
            "baseline_usable": base_usable,
            "adapter_usable": adapter_usable,
        })

    df = pd.DataFrame(paired_rows)
    n = len(df)
    if n == 0:
        summary = {
            "run": run_name,
            "n_common": 0,
            "baseline_accuracy": 0.0,
            "adapter_accuracy": 0.0,
            "accuracy_delta": 0.0,
            "baseline_usable_rate": 0.0,
            "adapter_usable_rate": 0.0,
            "usable_delta": 0.0,
            "adapter_only": 0,
            "baseline_only": 0,
            "net_correct_gain": 0,
        }
        return summary, df

    outcome_counts = df["outcome"].value_counts().to_dict()
    baseline_accuracy = df["baseline_correct"].mean()
    adapter_accuracy = df["adapter_correct"].mean()
    baseline_usable_rate = df["baseline_usable"].mean()
    adapter_usable_rate = df["adapter_usable"].mean()
    adapter_only = int(outcome_counts.get("adapter_only", 0))
    baseline_only = int(outcome_counts.get("baseline_only", 0))

    summary = {
        "run": run_name,
        "n_common": n,
        "baseline_accuracy": baseline_accuracy,
        "adapter_accuracy": adapter_accuracy,
        "accuracy_delta": adapter_accuracy - baseline_accuracy,
        "baseline_format_valid_rate": df["baseline_format_valid"].mean(),
        "adapter_format_valid_rate": df["adapter_format_valid"].mean(),
        "baseline_usable_rate": baseline_usable_rate,
        "adapter_usable_rate": adapter_usable_rate,
        "usable_delta": adapter_usable_rate - baseline_usable_rate,
        "both_correct": int(outcome_counts.get("both_correct", 0)),
        "adapter_only": adapter_only,
        "baseline_only": baseline_only,
        "both_wrong": int(outcome_counts.get("both_wrong", 0)),
        "net_correct_gain": adapter_only - baseline_only,
    }
    return summary, df


paired_summaries = []
paired_details = {}

if not baseline_output.exists():
    print(f"Missing baseline output: {baseline_output}")
else:
    for cfg in TRAIN_CONFIGS:
        name = cfg["name"]
        adapter_output = RUNS_DIR / f"{name}_outputs.jsonl"
        if not adapter_output.exists():
            continue

        summary, pair_df = paired_compare_outputs(baseline_output, adapter_output, name)
        paired_summaries.append(summary)
        paired_details[name] = pair_df

    if paired_summaries:
        paired_summary_df = pd.DataFrame(paired_summaries).sort_values(
            by=["net_correct_gain", "accuracy_delta", "usable_delta"],
            ascending=False,
        )
        display(paired_summary_df)
        paired_summary_df.to_csv(RUNS_DIR / "paired_compare.csv", index=False)
        print(f"Saved paired comparison to {RUNS_DIR / 'paired_compare.csv'}")

        best_run = paired_summary_df.iloc[0]["run"]
        print("Best paired run:", best_run)

        best_pairs = paired_details[best_run]
        fixed = best_pairs[best_pairs["outcome"] == "adapter_only"]
        regressed = best_pairs[best_pairs["outcome"] == "baseline_only"]

        print()
        print("Examples fixed by adapter:")
        display(fixed[["id", "ground_truth", "baseline_answer", "adapter_answer", "question"]].head(5))

        print()
        print("Examples regressed by adapter:")
        display(regressed[["id", "ground_truth", "baseline_answer", "adapter_answer", "question"]].head(5))
    else:
        print("No adapter output files found yet. Train/evaluate at least one adapter first.")


## 12. Download Outputs in Colab

Run this at the end of a Colab session to download metrics and sampled outputs.

In [ ]:
# Optional Colab download.
# import shutil
# archive = shutil.make_archive("student_sft_results", "zip", RUNS_DIR)
# from google.colab import files
# files.download(archive)